# Two-qubit RY sensing: manual angle sweep & optimization path

This tutorial mirrors the single-photon **1-qubit** tutorial, but with **two qubits**, each carrying a
single trainable `RY` gate in the setup (encoding) and decoding circuits. We:

1. build a two-qubit transient single-photon detection experiment,
2. optimize the four `RY` angles just like the 1-qubit case,
3. **skip the prebuilt `experiment.sweep`** and instead run a *manual*, fully parametric sweep over the
   two `RY` angles acting on **the first qubit** (the setup angle and the decoding angle),
4. overlay the optimizer's trajectory through those two angles on top of the swept landscape.

The manual sweep is a plain nested loop, so you can change the ranges, resolution and averaging by
editing a few variables in the sweep-configuration block.

In [22]:
from qsopt import *
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import qutip

print(f"JAX version: {jax.__version__}")
print(f"QuTiP version: {qutip.__version__}")
print(f"JAX backend: {jax.default_backend()}")
devices = jax.devices()
print(f"JAX devices: {devices}")
print(f"JAX GPU available: {any(d.platform == 'gpu' for d in devices)}")

JAX version: 0.9.2
QuTiP version: 5.2.2
JAX backend: cpu
JAX devices: [CpuDevice(id=0)]
JAX GPU available: False


## Define experimental parameters

Two qubits are dispersively coupled to the **same** cavity, each with its own coupling `chi`. Giving the two qubits *different* `chi` makes the first-qubit angle landscape non-trivial (the two qubits are not interchangeable).

In [ ]:
sigma = 1
k = 15 * sigma

from jax.scipy.special import erfc

def pulse(t, **kwargs):
    """
    Time-dependent coupling function for input cavity transparency.

    Args:
        t: float or JAX array, time variable
        **kwargs: Dictionary containing 'sigma' parameter (pulse bandwidth)

    Returns:
        JAX array: Normalized coupling strength g(t)
    """
    sigma = kwargs.get("sigma", 1.0)
    # exp(-x^2)/erfc(x) is a numerical 0/0 for large x (both underflow) -> NaN. Clamp |x|<=8 so the
    # coupling stays finite far past the pulse without changing anything in the physical region.
    dx = jnp.clip(sigma * t, -8.0, 8.0)
    coupling = jnp.sqrt(2 * sigma / jnp.sqrt(jnp.pi) * jnp.exp(-(dx**2)) / erfc(dx))
    return jnp.array(coupling, float)

# Two dispersive couplings: one per qubit, both to cavity 0, with different chi so the qubits differ.
interactions = [
    Interaction(interaction_type=InteractionType.DISPERSIVE,
                subsystem1=('cavity', 0), subsystem2=('qubit', 0),
                parameters={'chi': 0.5 * k}),
    Interaction(interaction_type=InteractionType.DISPERSIVE,
                subsystem1=('cavity', 0), subsystem2=('qubit', 1),
                parameters={'chi': 0.5 * k}),
    Interaction(interaction_type=InteractionType.XX,
                subsystem1=('qubit', 0), subsystem2=('qubit', 1),
                parameters={'chi': 0.1}),
]

# Two-qubit physical model
physical_model = PhysicalModel(
    perturbation_type='transient',
    n_cavities=1,
    n_fields=1,
    n_qubits=2,
    cavity_levels=2,
    field_levels=2,
    qubit_levels=2,
    interactions=interactions,
)

# Noise model (per-qubit rates broadcast automatically)
noise = NoiseModel(depolarizing=0.01, dephasing=0.01, relaxation=0.01)

# Time protocol
custom_times = TimeProtocol(
    t_simulation_start=-6,
    n_measurements=4,
    time_interval=4,
    random_measurements_offset=True,
    noisy_simulation_start=True,
)

config_set = [
    SystemConfiguration(
        name='no interaction',
        init_field_states={0: SubsystemState(State.FOCK, {'n': 0})},
        is_ground=True),
    SystemConfiguration(
        name='with interaction',
        init_field_states={0: SubsystemState(State.FOCK, {'n': 1})},
        interactions=[
            Interaction(interaction_type=InteractionType.INPUT_OUTPUT,
                        subsystem1=('cavity', 0), subsystem2=('field', 0),
                        parameters={'gamma': 1, 'kappa': k, 'sigma': sigma},
                        time_modulation=pulse),
        ]),
]

exp_parameters = ExperimentalParameters(
    physical_model=physical_model,
    noise_model=noise,
    time_protocol=custom_times,
    configuration_set=config_set,
)

print(exp_parameters)

SYSTEM DIMENSIONS
  Cavities: 1   levels [2]
  Fields:   1   levels [2]
  Qubits:   2   levels [2, 2]
  Total dimension: 16

PHYSICAL MODEL
  Perturbation type:    transient
  Interactions:         9 interaction(s)
  Static interactions:
    dispersive(cavity0-qubit0): chi=7.5
    dispersive(cavity0-qubit1): chi=7.5
    sx-sx(qubit0-qubit1): chi=0.1
    depolarizing(qubit0): depolarizing=0.01
    dephasing(qubit0): dephasing=0.01
    relaxation(qubit0): relaxation=0.01
    depolarizing(qubit1): depolarizing=0.01
    dephasing(qubit1): dephasing=0.01
    relaxation(qubit1): relaxation=0.01

TIME PROTOCOL
  Start time:            -6.0000
  Mode:                 Interval-based
  Time interval:          4.0000
  Number of measurements:      4
  Computed times:       [-2.0, 2.0, 6.0, 10.0]
  Collective offset:    uniform(-4, 0)
  Per-measurement jitter: none
  Noisy simulation start: on
  Measurement window:   None (uniform weights)

NOISE MODEL
  Depolarizing rate:    [0.01, 0.01]
  Dephas

## Define quantum circuits

A single `RY` per qubit in each circuit &mdash; four trainable angles in total. The parameter order is `[qubit 0, qubit 1]` inside each circuit, and the optimizer's flat vector is `[setup_q0, setup_q1, decode_q0, decode_q1]`.

In [24]:
from qsopt.core.circuit import create_ry_circuit

initial_circuit = create_ry_circuit(n_qubits=2, theta_values=[np.pi/2, np.pi/2])
final_circuit = create_ry_circuit(n_qubits=2, theta_values=[-np.pi/2, -np.pi/2])

print(f"Setup circuit parameters:    {initial_circuit.get_trainable_parameters()}")
print(initial_circuit)
print(f"Decoding circuit parameters: {final_circuit.get_trainable_parameters()}")
print(final_circuit)

Setup circuit parameters:    [Array(1.57079633, dtype=float64), Array(1.57079633, dtype=float64)]
QuantumCircuit(2 qubits, 2 gates)
  0: RY[0](param=1.571)
  1: RY[1](param=1.571)
Decoding circuit parameters: [Array(-1.57079633, dtype=float64), Array(-1.57079633, dtype=float64)]
QuantumCircuit(2 qubits, 2 gates)
  0: RY[0](param=-1.571)
  1: RY[1](param=-1.571)


## Define the experiment

The detection metric maximises the computational distance between the *with-interaction* and *no-interaction* measurement outcomes &mdash; the same criterion used in the 1-qubit tutorial, here on a 2-qubit readout.

In [25]:
detection_metric = DetectionMetric(
    n_cavities=1,
    n_fields=1,
    n_qubits=2,
    config_names=['with interaction', 'no interaction'],
    detection_criterion='max computational distance',
    perturbation_type=physical_model.perturbation_type,
)
print(detection_metric)

experiment = Experiment(
    experimental_params=exp_parameters,
    initial_circuit=initial_circuit,
    final_circuit=final_circuit,
    detection_metric=detection_metric,
)


DetectionMetric:
maximize computational distance
softmax aggregation, transient



## Run the optimization

Gradient descent over the four `RY` angles, exactly as in the 1-qubit tutorial. We pass an explicit callback so its parameter history survives for the trajectory plot at the end.

In [ ]:
import optax

# Explicit callback so this history is not overwritten by later evaluations.
history = OptimizationCallback(save_every=1, save_best=True)

history = experiment.optimize_rotations(
    initial_values=[0.4, 0.5, -1.1, -1.0],  # [setup_q0, setup_q1, decode_q0, decode_q1]
    num_steps=200,
    verbose=True,
    verbose_step=25,
    batch_size=16,
    tolerance=1e-7,
    callback=history,
    optimizer=optax.adam(learning_rate=0.7),
    anneal_tolerances=1000,
)

Configuration:
    Max iterations: 200
    Batch size: 16
    Convergence tolerance: 1.00e-07
    Detection metric:
maximize computational distance
softmax aggregation, transient
    Trainable parameters: 4 (2 initial circuit + 2 final circuit)
    Initial parameter values:
        param0.  setup_RY[0]  = 0.400  rad (22.9°)
        param1.  setup_RY[1]  = 0.500  rad (28.6°)
        param2.  reset_RY[0]  = -1.100 rad (-63.0°)
        param3.  reset_RY[1]  = -1.000 rad (-57.3°)
    Measurement uncertainty: collective offset uniform(-4, 0), per-measurement jitter off
Step  setup0_RY[0]   setup1_RY[1]   reset0_RY[0]   reset1_RY[1]   Metric      Validation  Grad Norm   Time
---------------------------------------------------------------------------------------------------------------


In [ ]:
# Free JAX memory after optimization
jax.clear_caches()
import gc; gc.collect()

NameError: name 'jax' is not defined

Build the deployable detection protocol at the best parameters (also loads them back onto the circuits).

In [ ]:
history = experiment.make_protocol(callback=history, batch_size=16)
print(history)

## Manual parametric sweep over the first-qubit angles

Instead of `experiment.sweep`, we sweep **by hand** over the two `RY` angles acting on **qubit 0**:
the *setup* angle and the *decoding* angle. For every point on the grid we set those two angles on the
circuits and call `run_simulation`, reading back the detection metric.

The two qubit-1 angles are held **fixed at their optimized values**, so the 2-D map is a slice of the
full 4-D landscape through the optimum. Everything below is a plain nested loop &mdash; change the ranges,
resolution and averaging in the configuration block to taste.

> This is the heavy cell: it runs `n_setup * n_reset` simulations. Start coarse and refine.

In [ ]:
# Qubit-1 angles are frozen at their optimized values while we sweep qubit 0.
best_init, bestnally, phonon-mediated kinetic inductance detectors show promising energy resolution for calorimetric applications \cite{hep3}.

% These experimental and theoretical results highlight the importance of developing highly efficient simulation and detection models for superconducting quantum devices. However, the main challenges in this direction lie in the presence of hardware noise, which degrades metrological performance, and the intrinsic mathematical complexity associated with the analytical study of open quantum systems, which complicates the identification of new optimal protocols.
% To address these two fundamental needs, we have developed Quantum Sensing Optimization (\qsopt) \footnote{https://github.com/Simone-Bordoni/Quantum-sensing-QML}. This framework leverages modern automatic differentiation techniques to efficiently optimize the parameters of existing protocols, making them robust to realistic noise conditions. Furthermore, we are employing this methodology to design and develop entirely novel detection protocols. Specifically, the framework allows for the training of the rotational parameters of a quantum circuit tasked with generating the pre-interaction state and decoding the post-interaction state. This functionality configures a small-scale Quantum Machin_final = history.get_best_trainable_params()
fixed_setup_q1 = float(best_init[1])
fixed_reset_q1 = float(best_final[1])
print(f"Frozen qubit-1 angles -> setup: {fixed_setup_q1:+.4f} rad, decoding: {fixed_reset_q1:+.4f} rad")

# ================= SWEEP CONFIGURATION (edit these) =================
n_setup = 15                    # grid resolution along the setup angle (qubit 0)
n_reset = 15                    # grid resolution along the decoding angle (qubit 0)
setup_range = (-np.pi, np.pi)   # setup angle range   (rad)
reset_range = (-np.pi, np.pi)   # decoding angle range (rad)
sweep_batch_size = 4            # timing realizations averaged per grid point
# ===================================================================

setup_grid = np.linspace(*setup_range, n_setup)
reset_grid = np.linspace(*reset_range, n_reset)


def manual_angle_sweep(experiment, setup_grid, reset_grid,
                       fixed_setup_q1, fixed_reset_q1, batch_size):
    """Sweep the first-qubit setup/decoding RY angles, returning the detection-metric grid.

    Args:
        experiment: the Experiment whose circuits are mutated in place.
        setup_grid (np.ndarray): setup-angle values for qubit 0 (x axis).
        reset_grid (np.ndarray): decoding-angle values for qubit 0 (y axis).
        fixed_setup_q1 (float): frozen setup angle for qubit 1.
        fixed_reset_q1 (float): frozen decoding angle for qubit 1.
        batch_size (int): timing realizations averaged per grid point.

    Returns:
        np.ndarray: metric grid of shape (len(reset_grid), len(setup_grid)).
    """
    metric_grid = np.zeros((len(reset_grid), len(setup_grid)))
    for i, r0 in enumerate(reset_grid):
        for j, s0 in enumerate(setup_grid):
            experiment.initial_circuit.set_trainable_parameters([float(s0), fixed_setup_q1])
            experiment.final_circuit.set_trainable_parameters([float(r0), fixed_reset_q1])
            cb = experiment.run_simulation(batch_size=batch_size)
            metric_grid[i, j] = cb.history['metric'][0]
        print(f"  decoding row {i + 1}/{len(reset_grid)} done")
    return metric_grid


metric_grid = manual_angle_sweep(
    experiment, setup_grid, reset_grid, fixed_setup_q1, fixed_reset_q1, sweep_batch_size)

# Restore the optimized parameters onto the circuits after the sweep.
experiment.initial_circuit.set_trainable_parameters(list(best_init))
experiment.final_circuit.set_trainable_parameters(list(best_final))

bi, bj = np.unravel_index(np.argmax(metric_grid), metric_grid.shape)
print(f"Sweep maximum {metric_grid[bi, bj]:.4f} at "
      f"setup={setup_grid[bj]:+.3f} rad, decoding={reset_grid[bi]:+.3f} rad")

## Visualize the optimization path over the swept landscape

We extract the optimizer's trajectory through the two first-qubit angles from the callback history and
draw it on top of the manually swept metric map. The path is a projection of the full 4-D trajectory onto
the qubit-0 plane, over a landscape sliced at the optimized qubit-1 angles &mdash; so the end of the path lands
near (but not exactly on) the swept maximum whenever the qubit-1 angles moved during training.

In [ ]:
# First-qubit angle trajectory: p = (initial_params, final_params); index [0] = qubit 0.
tp = history.history['trainable_params']
epochs = np.array(history.history['epochs'])
setup_q0_path = np.array([p[0][0] for p in tp])
reset_q0_path = np.array([p[1][0] for p in tp])

fig, ax = plt.subplots(figsize=(8.5, 7))

# Swept detection-metric landscape.
cf = ax.contourf(setup_grid, reset_grid, metric_grid, levels=30, cmap='viridis')
cbar = fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label('detection metric')

# Optimizer trajectory, colored by epoch.
ax.plot(setup_q0_path, reset_q0_path, '-', color='white', lw=1.0, alpha=0.6, zorder=2)
sc = ax.scatter(setup_q0_path, reset_q0_path, c=epochs, cmap='autumn',
                s=22, zorder=3, edgecolors='k', linewidths=0.3)

# Start, end and sweep-maximum markers.
ax.scatter(setup_q0_path[0], reset_q0_path[0], marker='o', s=150, facecolors='none',
           edgecolors='white', lw=2, zorder=4, label='start')
ax.scatter(setup_q0_path[-1], reset_q0_path[-1], marker='*', s=280, color='red',
           edgecolors='white', lw=0.8, zorder=4, label='optimized end')
ax.scatter(setup_grid[bj], reset_grid[bi], marker='X', s=150, color='cyan',
           edgecolors='k', lw=0.6, zorder=4, label='sweep maximum')

cbar_e = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.12)
cbar_e.set_label('optimization epoch')

ax.set_xlabel('setup RY angle on qubit 0  (rad)')
ax.set_ylabel('decoding RY angle on qubit 0  (rad)')
ax.set_title('Optimization path over the first-qubit angle landscape\n'
             '(qubit-1 angles frozen at their optimized values)')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig('results/two_qubit_angle_sweep_path.pdf', bbox_inches='tight')
plt.show()